In [1]:

# ============================================================
# RECOMENDADOR DE PLANES DE CAFE
# Sistema de recomendacion basado en habitos y consumo
# ============================================================

PLANES = {
    "maximo": "Mantenimiento Maximo",
    "moderado": "Mantenimiento Moderado",
    "minimo": "Mantenimiento Minimo",
    "descafeinado": "Descafeinado",
    "reducir": "Plan Reducir",
    "oficina": "Plan Oficina",
    "sueno": "Plan Sueno Sensible",
    "noche": "Plan Turno de Noche"
}

CAFEINA_POR_TAZA = 90
LIMITE_CAFEINA = 400


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def preguntar(texto, opciones):
    """Muestra una pregunta y valida la respuesta."""

    print("\n" + texto)

    for numero, opcion in enumerate(opciones, 1):
        print(f"{numero}. {opcion}")

    while True:
        try:
            respuesta = int(input("Selecciona una opcion: "))

            if 1 <= respuesta <= len(opciones):
                return respuesta

            print("Selecciona un numero valido.")

        except ValueError:
            print("Introduce un numero.")


def preguntar_si_no(texto):
    respuesta = preguntar(texto, ["Si", "No"])
    return respuesta == 1


def calcular_consumo():
    """Calcula la cafeina diaria segun las tazas de cafe."""

    print("\n--- CALCULO DE CONSUMO DE CAFEINA ---")
    print("Cada taza de cafe se estima en 90 mg de cafeina.")

    while True:
        try:
            tazas = int(
                input("¿Cuantas tazas de cafe consumes al dia? ")
            )

            if tazas < 0:
                print("No se admiten cantidades negativas.")
                continue

            total = tazas * CAFEINA_POR_TAZA

            print(f"\nTazas consumidas: {tazas}")
            print(f"Cafeina estimada: {total} mg/dia")

            return total

        except ValueError:
            print("Introduce un numero entero valido.")


# ============================================================
# 1. ENCUESTA
# ============================================================

def realizar_encuesta():

    print("\n======================================")
    print("       ENCUENTRA TU CAFE IDEAL")
    print("======================================")

    usuario = {}

    # SEGURIDAD: EDAD

    usuario["edad"] = preguntar(
        "¿Eres mayor de edad?",
        [
            "Si, soy mayor de 18 anos",
            "No, soy menor de 18 anos"
        ]
    )

    # SEGURIDAD: EMBARAZO Y LACTANCIA

    usuario["embarazo_lactancia"] = preguntar_si_no(
        "¿Estas embarazada o en periodo de lactancia?"
    )

    # SEGURIDAD: ENFERMEDADES Y CONDICIONES MEDICAS

    usuario["condicion_medica"] = preguntar(
        "¿Tienes alguna enfermedad o condicion medica "
        "por la que te hayan recomendado evitar o limitar "
        "el consumo de cafe o cafeina?",
        [
            "Si",
            "No",
            "No lo se"
        ]
    )

    # CONSUMO ACTUAL

    usuario["consumo"] = calcular_consumo()

    # OBJETIVOS

    usuario["objetivo"] = preguntar(
        "¿Cual es tu principal objetivo?",
        [
            "Mantener mi consumo actual",
            "Reducir progresivamente mi consumo",
            "Consumir la menor cantidad posible",
            "Eliminar la cafeina",
            "Tener energia durante mi jornada laboral",
            "Mejorar mi descanso"
        ]
    )

    usuario["dependencia"] = preguntar_si_no(
        "¿Sientes que necesitas cafe para funcionar "
        "o mantenerte despierto?"
    )

    # PREFERENCIAS

    usuario["preferencia"] = preguntar(
        "¿Que cantidad de cafeina te gustaria consumir?",
        [
            "Alta",
            "Moderada",
            "Baja",
            "Ninguna"
        ]
    )

    # SUENO Y SENSIBILIDAD

    usuario["sensibilidad"] = preguntar(
        "¿Como te afecta la cafeina?",
        [
            "No noto efectos negativos",
            "Me afecta ligeramente",
            "Me afecta bastante",
            "Me afecta mucho"
        ]
    )

    usuario["problemas_sueno"] = preguntar_si_no(
        "¿Tienes problemas frecuentes para dormir "
        "o sufres de insomnio?"
    )

    # ESTILO DE VIDA

    usuario["trabajo"] = preguntar(
        "¿Cual es tu situacion laboral habitual?",
        [
            "Trabajo en una oficina",
            "Trabajo desde casa",
            "Estudio",
            "Trabajo a turnos",
            "Trabajo habitualmente de noche",
            "Otro"
        ]
    )

    usuario["cafe_rendimiento"] = preguntar_si_no(
        "¿Utilizas el cafe principalmente para "
        "mantenerte despierto o concentrado?"
    )

    return usuario


# ============================================================
# 2. SISTEMA DE PUNTUACION
# ============================================================

def calcular_puntuaciones(usuario):

    puntos = {plan: 0 for plan in PLANES}

    motivos = {plan: [] for plan in PLANES}

    def sumar(plan, cantidad, motivo):
        puntos[plan] += cantidad
        motivos[plan].append(motivo)

    # TURNO DE NOCHE

    if usuario["trabajo"] == 5:
        sumar(
            "noche", 10,
            "Trabajas habitualmente durante la noche."
        )

    # SUENO SENSIBLE

    if usuario["problemas_sueno"]:
        sumar(
            "sueno", 10,
            "Has indicado que tienes problemas de sueno."
        )

    if usuario["sensibilidad"] == 3:
        sumar(
            "sueno", 3,
            "La cafeina te afecta bastante."
        )

    elif usuario["sensibilidad"] == 4:
        sumar(
            "sueno", 6,
            "La cafeina te afecta mucho."
        )

    elif usuario["sensibilidad"] == 2:
        sumar(
            "sueno", 1,
            "Presentas cierta sensibilidad a la cafeina."
        )

    # PLAN REDUCIR

    if usuario["objetivo"] == 2:
        sumar(
            "reducir", 10,
            "Quieres reducir progresivamente tu consumo."
        )

    if usuario["dependencia"]:
        sumar(
            "reducir", 5,
            "Sientes que necesitas cafe para funcionar."
        )

    if usuario["consumo"] > 400:
        sumar(
            "reducir", 8,
            "Tu consumo estimado supera los 400 mg diarios."
        )

    elif usuario["consumo"] >= 360:
        sumar(
            "reducir", 4,
            "Tu consumo se encuentra cerca de la referencia "
            "general de 400 mg diarios."
        )

    elif usuario["consumo"] >= 270:
        sumar(
            "reducir", 2,
            "Tu consumo estimado es relativamente elevado."
        )

    # PLAN OFICINA

    if usuario["trabajo"] == 1 and usuario["cafe_rendimiento"]:
        sumar(
            "oficina", 10,
            "Trabajas en oficina y utilizas cafe para rendir."
        )

    # DESCafeinado

    if usuario["objetivo"] == 4:
        sumar(
            "descafeinado", 10,
            "Quieres eliminar la cafeina de tu rutina."
        )

    # MANTENIMIENTO

    adulto_apto = (
        usuario["edad"] == 1
        and not usuario["embarazo_lactancia"]
    )

    if usuario["preferencia"] == 1:

        if adulto_apto and usuario["consumo"] <= 400:
            sumar(
                "maximo", 8,
                "Prefieres mantener un consumo elevado."
            )
        else:
            sumar(
                "moderado", 2,
                "No se recomienda asignar automaticamente "
                "el perfil de mantenimiento maximo."
            )

    elif usuario["preferencia"] == 2:
        sumar(
            "moderado", 8,
            "Prefieres un consumo moderado."
        )

    elif usuario["preferencia"] == 3:
        sumar(
            "minimo", 8,
            "Prefieres consumir poca cafeina."
        )

    elif usuario["preferencia"] == 4:
        sumar(
            "descafeinado", 10,
            "Prefieres eliminar la cafeina."
        )

    # OBJETIVOS ADICIONALES

    if usuario["objetivo"] == 3:
        sumar(
            "minimo", 5,
            "Quieres consumir la menor cantidad posible."
        )

    if usuario["objetivo"] == 5:
        sumar(
            "sueno", 5,
            "Tu prioridad es mejorar el descanso."
        )

    if usuario["objetivo"] == 6:
        sumar(
            "sueno", 5,
            "Quieres evitar que la cafeina afecte a tu descanso."
        )

    return puntos, motivos


# ============================================================
# 3. RECOMENDACION DE PLANES
# ============================================================

def recomendar_planes(usuario, puntos, motivos):

    # FILTRO DE SEGURIDAD

    if (
        usuario["edad"] == 2
        or usuario["embarazo_lactancia"]
        or usuario["condicion_medica"] in [1, 3]
    ):

        print("\n======================================")
        print("       RECOMENDACION LIMITADA")
        print("======================================")

        if usuario["edad"] == 2:
            print(
                "\nLa persona usuaria es menor de edad."
            )

        if usuario["embarazo_lactancia"]:
            print(
                "\nHas indicado que estas embarazada "
                "o en periodo de lactancia."
            )

        if usuario["condicion_medica"] == 1:
            print(
                "\nHas indicado que tienes una enfermedad "
                "o condicion medica por la que te han "
                "recomendado evitar o limitar la cafeina."
            )

        if usuario["condicion_medica"] == 3:
            print(
                "\nNo tienes claro si tu condicion medica "
                "puede afectar al consumo de cafeina."
            )

        print(
            "\nNo podemos generar una recomendacion "
            "personalizada de consumo de cafeina "
            "para este perfil."
        )

        print(
            "\nConsulta con un profesional sanitario "
            "para establecer una pauta adecuada."
        )

        print(
            f"\nConsumo estimado: {usuario['consumo']} mg/dia"
        )

        return None, None

    # ORDEN DE PRIORIDAD EN CASO DE EMPATE

    orden = [
        "noche",
        "sueno",
        "descafeinado",
        "reducir",
        "oficina",
        "minimo",
        "moderado",
        "maximo"
    ]

    ranking = sorted(
        puntos.keys(),
        key=lambda plan: (
            -puntos[plan],
            orden.index(plan)
        )
    )

    # DETERMINAR SI NECESITA REDUCIR

    necesita_reducir = (
        usuario["objetivo"] == 2
        or usuario["consumo"] > 400
        or ranking[0] == "reducir"
    )

    # DETERMINAR EL PLAN DE DESTINO

    if usuario["objetivo"] == 4 or usuario["preferencia"] == 4:

        destino = "descafeinado"

    elif usuario["trabajo"] == 5:

        destino = "noche"

    elif usuario["problemas_sueno"]:

        destino = "sueno"

    elif usuario["trabajo"] == 1 and usuario["cafe_rendimiento"]:

        destino = "oficina"

    else:

        destinos_posibles = [
            plan for plan in ranking
            if plan != "reducir"
        ]

        destino = destinos_posibles[0]

    # ASIGNAR PLANES

    if necesita_reducir:

        principal = "reducir"
        secundario = destino

    else:

        principal = destino

        # PLAN COMPLEMENTARIO

        secundarios = [
            plan for plan in ranking
            if plan != principal
            and plan != "reducir"
            and puntos[plan] > 0
        ]

        secundario = (
            secundarios[0]
            if secundarios
            else None
        )

    return principal, secundario


# ============================================================
# 4. RESULTADO PERSONALIZADO
# ============================================================

def mostrar_resultado(usuario, principal, secundario, puntos, motivos):

    print("\n======================================")
    print("          TU RECOMENDACION")
    print("======================================")

    if principal is None:
        return

    print(f"\nPLAN PRINCIPAL: {PLANES[principal]}")

    if secundario:
        print(
            f"PLAN COMPLEMENTARIO: {PLANES[secundario]}"
        )

    print("\n--- TU PERFIL ---")

    print(
        f"Consumo estimado: {usuario['consumo']} mg/dia"
    )

    print(
        f"Equivalencia: "
        f"{usuario['consumo'] // CAFEINA_POR_TAZA} tazas de cafe"
    )

    print(
        f"Dependencia percibida: "
        f"{'Si' if usuario['dependencia'] else 'No'}"
    )

    print(
        f"Problemas de sueno: "
        f"{'Si' if usuario['problemas_sueno'] else 'No'}"
    )

    print("\n--- ¿POR QUE TE RECOMENDAMOS ESTE PLAN? ---")

    for motivo in motivos[principal]:
        print(f"- {motivo}")

    if secundario:
        print(
            f"\nUna vez reducido tu consumo, "
            f"tu plan de destino recomendado seria: "
            f"{PLANES[secundario]}."
        )

        if secundario != principal and motivos[secundario]:
            print("\nMotivos del plan de destino:")

            for motivo in motivos[secundario]:
                print(f"- {motivo}")

    print("\n--- PUNTUACIONES ---")

    for plan, puntuacion in sorted(
        puntos.items(),
        key=lambda elemento: elemento[1],
        reverse=True
    ):
        print(
            f"{PLANES[plan]}: {puntuacion} puntos"
        )

    # AVISOS DE CONSUMO

    if usuario["consumo"] > LIMITE_CAFEINA:
        print("\nATENCION:")
        print(
            "Tu consumo estimado supera los 400 mg diarios. "
            "Esta cantidad supera la referencia general "
            "de seguridad para adultos sanos."
        )

    elif usuario["consumo"] == LIMITE_CAFEINA:
        print("\nATENCION:")
        print(
            "Tu consumo estimado alcanza los 400 mg diarios, "
            "la referencia general para adultos sanos."
        )

    elif usuario["consumo"] >= 360:
        print("\nAVISO:")
        print(
            "Tu consumo se encuentra cerca de los 400 mg diarios."
        )

    if usuario["consumo"] > 200:
        print(
            "Revisa tambien la distribucion de la cafeina "
            "a lo largo del dia y su efecto sobre tu descanso."
        )

    print("\n======================================")


# ============================================================
# 5. EJECUCION DEL PROGRAMA
# ============================================================

def main():

    usuario = realizar_encuesta()

    puntos, motivos = calcular_puntuaciones(usuario)

    principal, secundario = recomendar_planes(
        usuario,
        puntos,
        motivos
    )

    mostrar_resultado(
        usuario,
        principal,
        secundario,
        puntos,
        motivos
    )


if __name__ == "__main__":
    main()



       ENCUENTRA TU CAFE IDEAL

¿Eres mayor de edad?
1. Si, soy mayor de 18 anos
2. No, soy menor de 18 anos
Selecciona una opcion: 2

¿Estas embarazada o en periodo de lactancia?
1. Si
2. No
Selecciona una opcion: 2

¿Tienes alguna enfermedad o condicion medica por la que te hayan recomendado evitar o limitar el consumo de cafe o cafeina?
1. Si
2. No
3. No lo se
Selecciona una opcion: 2

--- CALCULO DE CONSUMO DE CAFEINA ---
Cada taza de cafe se estima en 90 mg de cafeina.
¿Cuantas tazas de cafe consumes al dia? 6

Tazas consumidas: 6
Cafeina estimada: 540 mg/dia

¿Cual es tu principal objetivo?
1. Mantener mi consumo actual
2. Reducir progresivamente mi consumo
3. Consumir la menor cantidad posible
4. Eliminar la cafeina
5. Tener energia durante mi jornada laboral
6. Mejorar mi descanso
Selecciona una opcion: 3

¿Sientes que necesitas cafe para funcionar o mantenerte despierto?
1. Si
2. No
Selecciona una opcion: 1

¿Que cantidad de cafeina te gustaria consumir?
1. Alta
2. Moderada
3